In [0]:
CATALOG = "workspace"
SCHEMA = "incident_assistent"
print("Using schema:", SCHEMA)
TABLE_A = f"{CATALOG}.{SCHEMA}.chunks_small"
TABLE_B = f"{CATALOG}.{SCHEMA}.chunks_large"
print(spark.table(TABLE_A).count(), "small chunks")
print(spark.table(TABLE_B).count(), "large chunks")

Using schema: incident_assistent
63 small chunks
30 large chunks


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
for e in w.serving_endpoints.list():
    print(e.name)

databricks-gpt-oss-120b
databricks-gpt-oss-20b
databricks-qwen3-next-80b-a3b-instruct
databricks-qwen35-122b-a10b
databricks-gemma-3-12b
databricks-gte-large-en
databricks-bge-large-en
databricks-meta-llama-3-1-8b-instruct
databricks-meta-llama-3-3-70b-instruct
databricks-qwen3-embedding-0-6b
databricks-llama-4-maverick
databricks-deepseek-v4-flash-0731


In [0]:
EMBED_MODEL = "databricks-gte-large-en"

In [0]:
for ep in w.vector_search_endpoints.list_endpoints():
    print(ep.name, "->", ep.endpoint_status.state if ep.endpoint_status else None)

incident_vs_endpoint -> EndpointStatusState.ONLINE


In [0]:
for ep in w.vector_search_endpoints.list_endpoints():
    print(ep.name, "->", ep.endpoint_status.state if ep.endpoint_status else None)

incident_vs_endpoint -> EndpointStatusState.ONLINE


In [0]:
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest, EmbeddingSourceColumn, PipelineType, VectorIndexType)

ENDPOINT = "incident_vs_endpoint"
INDEX_A = f"{CATALOG}.{SCHEMA}.idx_small"
INDEX_B = f"{CATALOG}.{SCHEMA}.idx_large"

def create_index(source_table, index_name):
    return w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=ENDPOINT,
        primary_key="chunk_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=source_table,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(name="text", embedding_model_endpoint_name=EMBED_MODEL)
            ],
        ),
    )

create_index(TABLE_A, INDEX_A)
create_index(TABLE_B, INDEX_B)
print("Indexes requested")

---------------------------------------------------------------------------
ResourceAlreadyExists                     Traceback (most recent call last)
File <command-7662139039053784>, line 23
      8 def create_index(source_table, index_name):
      9     return w.vector_search_indexes.create_index(
     10         name=index_name,
     11         endpoint_name=ENDPOINT,
   (...)     20         ),
     21     )
---> 23 create_index(TABLE_A, INDEX_A)
     24 create_index(TABLE_B, INDEX_B)
     25 print("Indexes requested")

File <command-7662139039053784>, line 9, in create_index(source_table, index_name)
      8 def create_index(source_table, index_name):
----> 9     return w.vector_search_indexes.create_index(
     10         name=index_name,
     11         endpoint_name=ENDPOINT,
     12         primary_key="chunk_id",
     13         index_type=VectorIndexType.DELTA_SYNC,
     14         delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
     15             source_table=source

In [0]:
for name in [INDEX_A, INDEX_B]:
    idx = w.vector_search_indexes.get_index(name)
    print(name, "-> ready:", idx.status.ready, "|", idx.status.message)

workspace.incident_assistent.idx_small -> ready: True | Index creation succeeded. Check latest status: https://dbc-a45451e5-aa32.cloud.databricks.com/explore/data/workspace/incident_assistent/idx_small
workspace.incident_assistent.idx_large -> ready: True | Index creation succeeded. Check latest status: https://dbc-a45451e5-aa32.cloud.databricks.com/explore/data/workspace/incident_assistent/idx_large


In [0]:
res = w.vector_search_indexes.query_index(
    index_name=INDEX_A,
    columns=["chunk_id", "doc_id", "text"],
    query_text="My Spark job failed with out of memory",
    num_results=3,
)
for row in res.result.data_array:
    print(row[1], "->", row[2][:120], "\n")

data_skew -> Severe Data Skew Causing Task Hangs and OOM Symptoms: Spark job hangs at 99% completion on a single remaining task, or t 

out_of_memory_error -> Driver Out-of-Memory Error During Collect Operation Symptoms: PySpark job crashes with java.lang.OutOfMemoryError: Java  

shuffle_spill_to_disk -> Shuffle Spill to Disk Degrading Spark Performance Symptoms: Databricks Spark job runs significantly slower than normal,  



In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

INDEX_A = f"{CATALOG}.{SCHEMA}.idx_small"
INDEX_B = f"{CATALOG}.{SCHEMA}.idx_large"

In [0]:
for n in [INDEX_A, INDEX_B]:
    print(n, "-> ready:", w.vector_search_indexes.get_index(n).status.ready)

workspace.incident_assistent.idx_small -> ready: True
workspace.incident_assistent.idx_large -> ready: True


In [0]:
def search(question, index_name, k=3):
    res = w.vector_search_indexes.query_index(
        index_name=index_name,
        columns=["chunk_id", "doc_id", "text"],
        query_text=question,
        num_results=k,
    )
    return res.result.data_array   # each row: [chunk_id, doc_id, text, score]

for q in ["How do I fix data skew in a join?",
          "Pipeline ran twice and created duplicates",
          "Delta merge fails with a concurrent update"]:
    print("\nQUESTION:", q)
    for row in search(q, INDEX_A):
        print("  ->", row[1])


QUESTION: How do I fix data skew in a join?
  -> data_skew
  -> null_values_breaking_a_join
  -> slow_join

QUESTION: Pipeline ran twice and created duplicates
  -> job_running_twice
  -> duplicate_records
  -> job_running_twice

QUESTION: Delta merge fails with a concurrent update
  -> delta_merge_conflict
  -> delta_merge_conflict
  -> concurrent_write_error


In [0]:
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

LLM = "databricks-meta-llama-3-3-70b-instruct"

def call_llm(prompt, max_tokens=400):
    resp = w.serving_endpoints.query(
        name=LLM,
        messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
        max_tokens=max_tokens,
        temperature=0.1,
    )
    return resp.choices[0].message.content

print(call_llm("Say hello in one short sentence."))

Hello.


In [0]:
def ask(question, index_name, k=3):
    hits = search(question, index_name, k)
    context = "\n\n".join(f"[{h[1]}] {h[2]}" for h in hits)

    prompt = f"""You are a data pipeline support assistant.
Answer ONLY using the context below.
If the answer is not in the context, say "I don't know".
At the end, list the source ids in square brackets.

Context:
{context}

Question: {question}"""

    return call_llm(prompt), [h[1] for h in hits]

answer, sources = ask("How do I fix data skew in a join?", INDEX_A)
print(answer)
print("\nSources:", sources)

To fix data skew in a join, you can follow these steps: 
1. Identify the skewed key using df.groupBy('key').count().orderBy(col('count').desc()).show(). 
2. Filter out or isolate dummy/null keys before performing joins. 
3. Apply a salt technique: append a random integer suffix to the skewed key on one side and expand the joining dataframe on the other side. 
4. Alternatively, enable AQE (Adaptive Query Execution) by setting spark.sql.adaptive.enabled = true and spark.sql.adaptive.skewJoin.enabled = true.

[data_skew]

Sources: ['data_skew', 'null_values_breaking_a_join', 'slow_join']


In [0]:
tests = [
    "How do I fix data skew in a join?",            # should be answerable
    "What is the capital of France?",               # not in the runbooks
    "My job is slow and shuffle is spilling to disk",  # reworded question
]
for q in tests:
    a, s = ask(q, INDEX_A)
    print("\nQ:", q, "\nA:", a, "\nSources:", s)


Q: How do I fix data skew in a join? 
A: To fix data skew in a join, you can follow these steps: 
1. Identify the skewed key using df.groupBy('key').count().orderBy(col('count').desc()).show(). 
2. Filter out or isolate dummy/null keys before performing joins. 
3. Apply a salt technique: append a random integer suffix to the skewed key on one side and expand the joining dataframe on the other side. 
4. Alternatively, enable AQE (Adaptive Query Execution) by setting spark.sql.adaptive.enabled = true and spark.sql.adaptive.skewJoin.enabled = true.

[data_skew] 
Sources: ['data_skew', 'null_values_breaking_a_join', 'slow_join']

Q: What is the capital of France? 
A: I don't know [data_skew, job_running_twice, delta_merge_conflict] 
Sources: ['data_skew', 'job_running_twice', 'delta_merge_conflict']

Q: My job is slow and shuffle is spilling to disk 
A: To fix the issue of shuffle spilling to disk, you can try the following steps: 
1. Scale up the Databricks cluster worker nodes to a memo

In [0]:
q = "How do I fix data skew in a join?"
print("SMALL:", [h[1] for h in search(q, INDEX_A)])
print("LARGE:", [h[1] for h in search(q, INDEX_B)])

SMALL: ['data_skew', 'null_values_breaking_a_join', 'slow_join']
LARGE: ['null_values_breaking_a_join', 'wrong_data_types_after_join', 'slow_join']
